In [1]:
import os
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [ ]:
# Google API key environment variable থেকে নিচ্ছি

google_api_key = os.getenv("GOOGLE_API_KEY")

In [3]:
# local embedding model import

MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8203.51it/s]


In [4]:
documents = [
    Document(page_content="LangChain is a framework for LLM applications."),
    Document(page_content="Retrieval augmented generation improves answer accuracy."),
    Document(page_content="Vector stores store text as embeddings."),
    Document(page_content="Chroma is a popular local vector database."), 
    Document(page_content="Python is a programming language") 
]

vector_store = Chroma(
    collection_name="multi_query_rectriever_doc",
    embedding_function=embeddings,
    persist_directory="../all_vector_stores/chroma_multi_query_rectriever"
)

vector_store.add_documents(documents)

['b99cb6c9-3dfb-4d5e-bf1f-ec6cc602d408',
 '57f47715-e680-4824-8645-ef0bbe4a4207',
 '0c225cc9-926c-4f3b-a4d1-49d80d5e30bb',
 '066bd803-4f5c-4b3f-939c-edee284f0cfc',
 '415f0137-a4de-4357-acbc-749399cb09e6']

In [ ]:
# Base retriever তৈরি করছি

base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Google free model
llm = ChatGoogleGenerativeAI(
    model="gemini-4-hydrogen",
    google_api_key=google_api_key,
    temperature=0
)



In [ ]:
# MultiQueryRetriever তৈরি করছি. এটি LLM দিয়ে original query-র কয়েকটি alternative query বানাবে

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

In [7]:
query = "How can I improve RAG answer quality?"

results = multi_query_retriever.invoke(query)

print("Original Query:", query)
print("-" * 50)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Original Query: How can I improve RAG answer quality?
--------------------------------------------------
1. Retrieval augmented generation improves answer accuracy.
2. Chroma is a popular local vector database.
3. Vector stores store text as embeddings.
4. LangChain is a framework for LLM applications.
